# SoundStream Demo

Runs my SoundStream codec on whatever audio URL you paste in. Default points
at the LJ Speech clip from the assignment, swap it for anything you want — I
downmix to mono and resample to 16 kHz inside the notebook anyway, so the
source format doesn't matter.

Repo: https://github.com/tolyaho/NeuralAudioCodec — training, eval and design
notes are all there.

## Setup

Clones the repo, grabs `soundfile` (Colab has the rest) and pulls the 213 MB
checkpoint from HF. Safe to re-run: clone is reused, checkpoint skipped if
it's already on disk.

In [ ]:
import os
if not os.path.isdir('NeuralAudioCodec'):
    !git clone -q https://github.com/tolyaho/NeuralAudioCodec.git
%cd NeuralAudioCodec

In [ ]:
!pip install -q soundfile

In [ ]:
!bash scripts/download_checkpoint.sh

## Audio in

Point `AUDIO_URL` at whatever you want to reconstruct. WAV just works; for
MP3 / M4A drop a `!apt-get install -y ffmpeg` cell above. The cell after
pulls the bytes, downmixes to mono, resamples to 16 kHz.

In [ ]:
AUDIO_URL = 'https://keithito.com/LJ-Speech-Dataset/LJ025-0076.wav'

In [ ]:
import io
import requests
import torch
import torchaudio

SAMPLE_RATE = 16000

response = requests.get(AUDIO_URL, timeout=30)
response.raise_for_status()

waveform, sr = torchaudio.load(io.BytesIO(response.content))

if waveform.shape[0] > 1:
    waveform = waveform.mean(dim=0, keepdim=True)
if sr != SAMPLE_RATE:
    waveform = torchaudio.functional.resample(waveform, sr, SAMPLE_RATE)

print(f'loaded {waveform.shape[-1] / SAMPLE_RATE:.2f} s, {SAMPLE_RATE} Hz, mono')

## Codec

Same SoundStream I trained — 32 base channels, 512-dim latent, 1024-entry
codebook × 8 residual quantizers, encoder/decoder strides `[2, 4, 5, 5]`.
Weights come from the checkpoint above. The EMA-buffer sync is a no-op on
this final checkpoint; it's there as a safety net for older runs that didn't
save those buffers.

In [ ]:
import sys
sys.path.insert(0, '.')

from src.model.soundstream import SoundStream

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = SoundStream(
    in_channels=1,
    base_channels=32,
    latent_dim=512,
    codebook_size=1024,
    num_quantizers=8,
).to(device).eval()

ckpt = torch.load('checkpoints/final_soundstream.pt', map_location=device)
state = ckpt['codec'] if 'codec' in ckpt else ckpt.get('model', ckpt)
missing, _ = model.load_state_dict(state, strict=False)

if any('ema_' in k for k in missing):
    for vq in model.quantizer.quantizers:
        vq.ema_cluster_sum.data.copy_(vq.codebook.weight.data)
        vq.ema_cluster_size.fill_(1.0)

print('loaded checkpoint at step', ckpt.get('step'))

## Forward pass

Whole clip through encoder → RVQ → decoder in one shot. I clamp the output
to `[-1, 1]` because the decoder occasionally pushes a sample slightly
outside that range on transients.

In [ ]:
with torch.no_grad():
    audio = waveform.unsqueeze(0).to(device)  # [1, 1, T]
    out = model(audio)
    reconstruction = out['reconstruction'].clamp(-1.0, 1.0)

print('input ', tuple(audio.shape))
print('output', tuple(reconstruction.shape))

## Listen

Top widget is the original at 16 kHz. Bottom is the codec reconstruction at
6.4 kbps — 80 frames/s × 8 quantizers × log₂(1024) bits.

In [ ]:
from IPython.display import Audio, display

original = audio.squeeze().cpu().numpy()
reconstructed = reconstruction.squeeze().cpu().numpy()

print('original')
display(Audio(original, rate=SAMPLE_RATE))
print('reconstructed')
display(Audio(reconstructed, rate=SAMPLE_RATE))